In [1]:
import os
import sys
import re
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

df = pd.read_csv('../DATASET/augmented.csv')

WINDOW_SIZE = 8
STEP_SIZE = 7

def create_windows(df, window_size, step_size):
    windows = []
    labels = []
    users = []
    
    # Updated logic: Use regex to match columns starting with x, y, or z followed by a number
    # ^[xyz]\d matches: start of string, then x or y or z, then any digit (0-9)
    feature_cols = [col for col in df.columns if re.match(r'^[xyz]\d', col)]

    for user_id in df['user'].unique():
        user_df = df[df['user'] == user_id].reset_index(drop=True)
        num_samples = len(user_df)

        for i in range(0, num_samples - window_size + 1, step_size):
            window = user_df.iloc[i:i + window_size]

            # Extract features (mean, std, min, max) for each sensor axis
            features = {}
            for col in feature_cols:
                features[f'{col}_mean'] = window[col].mean()
                features[f'{col}_std'] = window[col].std()
                features[f'{col}_min'] = window[col].min()
                features[f'{col}_max'] = window[col].max()

            windows.append(features)
            labels.append(window['class'].mode()[0]) # Majority class in the window
            users.append(user_id)

    windowed_df = pd.DataFrame(windows)
    windowed_df['class'] = labels
    windowed_df['user'] = users
    return windowed_df

# Create windowed_df
windowed_df = create_windows(df, WINDOW_SIZE, STEP_SIZE)

# Separate features (X_windowed_discretized) and target (y_windowed)
X_windowed_discretized = windowed_df.drop(['user', 'class'], axis=1)
y_windowed = windowed_df['class']

print("Shape of X_windowed_discretized:", X_windowed_discretized.shape)
print("Shape of y_windowed:", y_windowed.shape)
print("First 5 rows of windowed_df:")
display(windowed_df.head())

def project_root() -> Path:
    start = Path.cwd().resolve()
    return start.parents[1]

# Store it in project_root()/DATASET/windowed.csv
windowed_output_path = project_root() / 'NOTEBOOKS' / 'DATASET' / 'windowed.csv'
windowed_output_path.parent.mkdir(parents=True, exist_ok=True) # Ensure directory exists
windowed_df.to_csv(windowed_output_path, index=False)
print(f'Windowed dataset written to: {windowed_output_path.resolve()}')

Shape of X_windowed_discretized: (23660, 48)
Shape of y_windowed: (23660,)
First 5 rows of windowed_df:


,x1_mean,x1_std,x1_min,x1_max,y1_mean,y1_std,y1_min,y1_max,z1_mean,z1_std,...,y4_mean,y4_std,y4_min,y4_max,z4_mean,z4_std,z4_min,z4_max,class,user
0,-1.500,1.309307,-3,1,95.875,2.356602,92,100,-61.625,2.133910,...,-103.750,0.462910,-104,-103,-143.500,1.772811,-147,-142,sitting,debora
1,0.000,1.309307,-1,3,97.250,1.488048,95,100,-61.000,1.414214,...,-103.750,0.462910,-104,-103,-144.250,1.164965,-146,-142,sitting,debora
2,0.125,0.991031,-1,2,97.125,1.457738,95,99,-61.375,1.060660,...,-104.375,1.407886,-106,-102,-144.875,0.991031,-146,-143,sitting,debora
3,-0.625,2.503569,-4,3,97.500,1.927248,95,100,-62.125,0.834523,...,-103.500,6.023762,-112,-96,-140.375,3.700869,-145,-133,sitting,debora
4,-1.625,1.846812,-4,1,98.875,1.457738,96,100,-65.000,1.603567,...,-98.875,3.758324,-105,-95,-136.000,3.162278,-141,-133,sitting,debora


Windowed dataset written to: C:\Users\matte\Desktop\DMML_Wearable_Body_Postures\NOTEBOOKS\DATASET\windowed.csv
